# Fine-tuning pyannote on my voice + Turkish voices

Takes the cleaned chunks from `Audio_Data_Cleaning.ipynb`, adds other Turkish speakers, stitches both into fake conversations with who-spoke-when labels, and fine-tunes `pyannote/segmentation-3.0` on them.

Needs a Hugging Face login with the model terms accepted: https://huggingface.co/pyannote/segmentation-3.0

In [1]:
import csv
import io
import logging
import random
from pathlib import Path

import numpy as np
import soundfile as sf

# progress goes to the notebook and to a log file, so long steps can be watched from outside
logging.basicConfig(level=logging.WARNING, format="%(asctime)s %(message)s", datefmt="%H:%M:%S",
                    handlers=[logging.StreamHandler(), logging.FileHandler("fine_tuning.log")], force=True)
logger = logging.getLogger("fine_tuning")
logger.setLevel(logging.INFO)
log = logger.info

SR = 16000
SEED = 0
DATA = Path("data")
ME = DATA / "me"
OTHERS = DATA / "others"
CONV = DATA / "conversations"
TARGET_RMS = 10 ** (-23 / 20)


def normalize(x):
    """Same loudness for every clip (-23 dBFS RMS), without clipping."""
    x = x * (TARGET_RMS / np.sqrt(np.mean(x ** 2)))
    return x / max(1.0, np.abs(x).max() / 0.99)

## 1. Other Turkish voices

The Alania set is 540 GB, so only a slice is read: ~7,000 clips, about 10 hours. Where a clip has a simulated mic/room version (`audio_channel`) that one is used, because it is closer to a real microphone than the clean studio render.

In [2]:
import pyarrow.parquet as pq
from huggingface_hub import HfFileSystem
from scipy.signal import resample_poly

N_CLIPS = 7000
REPO = "datasets/cloud0day3/alania-synthetic-speech-tr/data/cc-by"
OTHERS_MANIFEST = DATA / "others_manifest.csv"

# ponytail: no resume, an interrupted download starts over. Add resume if the slice grows.
if not OTHERS_MANIFEST.exists():
    OTHERS.mkdir(parents=True, exist_ok=True)
    fs = HfFileSystem()
    rows = []
    for shard in sorted(fs.glob(f"{REPO}/*.parquet")):
        pf = pq.ParquetFile(fs.open(shard, "rb", block_size=8 * 2**20))
        for g in range(pf.num_row_groups):
            table = pf.read_row_group(g, columns=["id", "voice_id", "gender", "duration", "audio", "audio_channel"])
            for r in table.to_pylist():
                if not 1.5 <= r["duration"] <= 10:
                    continue
                clip = r["audio_channel"] or r["audio"]
                x, sr = sf.read(io.BytesIO(clip["bytes"]), dtype="float32")
                if x.ndim > 1:
                    x = x.mean(axis=1)
                x = normalize(resample_poly(x, SR, sr))
                name = f"{r['voice_id']}_{r['id']}.wav"
                sf.write(OTHERS / name, x, SR, subtype="PCM_16")
                rows.append((name, r["voice_id"], r["gender"], f"{len(x) / SR:.2f}"))
            log(f"others {len(rows)}/{N_CLIPS} clips")
            if len(rows) >= N_CLIPS:
                break
        if len(rows) >= N_CLIPS:
            break
    with open(OTHERS_MANIFEST, "w", newline="") as f:
        csv.writer(f).writerows([("file", "voice", "gender", "duration"), *rows])

others = list(csv.DictReader(open(OTHERS_MANIFEST)))
voices = sorted({r["voice"] for r in others})
hours = sum(float(r["duration"]) for r in others) / 3600
print(f"{len(others)} clips, {len(voices)} voices, {hours:.1f} h")

/home/darkwarro/Documents/Web_Apps/Exposure Academy/Academy_Exercises/Speaker_Diarization/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


7048 clips, 507 voices, 8.7 h


## 2. Fake conversations

A diarization model learns from conversations, not single clips. Each 60 s conversation has 2-3 speakers taking turns with random pauses, and sometimes talking over each other. Because we place every clip ourselves, the who-spoke-when labels are free.

Validation uses my held-out chunks and voices the model never sees in training.

In [3]:
CONV_S = 60
N_CONV = {"train": 800, "val": 40}
ME_PROB = 0.5       # share of conversations I am in
OVERLAP_PROB = 0.15

# pools: speaker -> list of clip paths, per split
pools = {"train": {}, "val": {}}
for r in csv.DictReader(open(DATA / "me_manifest.csv")):
    pools[r["split"]].setdefault("me", []).append(ME / r["file"])
shuffled = random.Random(SEED).sample(voices, len(voices))
val_voices = set(shuffled[: len(voices) * 15 // 100])
for r in others:
    split = "val" if r["voice"] in val_voices else "train"
    pools[split].setdefault(r["voice"], []).append(OTHERS / r["file"])


def make_conversation(rng, pool):
    """Mix clips into one conversation. Returns (audio, [(start, duration, speaker), ...])."""
    n_speakers = rng.choice([2, 3])
    speakers = ["me"] if rng.random() < ME_PROB else []
    speakers += rng.sample([s for s in pool if s != "me"], n_speakers - len(speakers))
    gain = {s: 10 ** (rng.uniform(-6, 3) / 20) for s in speakers}  # people sit at different distances

    mix = np.zeros(CONV_S * SR, dtype=np.float32)
    turns, t, prev = [], rng.uniform(0, 1), None
    while True:
        spk = rng.choice(speakers)
        clip, _ = sf.read(rng.choice(pool[spk]), dtype="float32")
        if spk != prev and prev is not None and rng.random() < OVERLAP_PROB:
            start = t - rng.uniform(0.2, 1.0)  # starts before the previous speaker finished
        else:
            start = t + rng.uniform(0.1, 1.0)
        i = int(start * SR)
        if i + len(clip) > len(mix):
            break
        mix[i:i + len(clip)] += clip * gain[spk]
        turns.append((i / SR, len(clip) / SR, spk))
        t, prev = (i + len(clip)) / SR, spk

    # faint noise floor, otherwise the gaps between clips are perfect digital silence
    noise = np.random.default_rng(rng.getrandbits(32)).normal(0, 10 ** (rng.uniform(-65, -50) / 20), len(mix))
    mix = mix + noise.astype(np.float32)
    return mix / max(1.0, np.abs(mix).max() / 0.99), turns


if not (CONV / "database.yml").exists():
    for sub in ("wav", "rttm", "uem", "lists"):
        (CONV / sub).mkdir(parents=True, exist_ok=True)
    rng = random.Random(SEED)
    for split, n in N_CONV.items():
        uris = []
        for k in range(n):
            uri = f"{split}_{k:04d}"
            mix, turns = make_conversation(rng, pools[split])
            sf.write(CONV / "wav" / f"{uri}.wav", mix, SR, subtype="PCM_16")
            (CONV / "rttm" / f"{uri}.rttm").write_text("".join(
                f"SPEAKER {uri} 1 {s:.3f} {d:.3f} <NA> <NA> {spk} <NA> <NA>\n" for s, d, spk in turns))
            (CONV / "uem" / f"{uri}.uem").write_text(f"{uri} 1 0.000 {CONV_S:.3f}\n")
            uris.append(uri)
            if (k + 1) % 100 == 0 or k + 1 == n:
                log(f"conversations {split} {k + 1}/{n}")
        (CONV / "lists" / f"{split}.txt").write_text("\n".join(uris) + "\n")

    # tells pyannote where the audio and labels are (paths are relative to this file)
    (CONV / "database.yml").write_text("""Databases:
  MeTR: wav/{uri}.wav

Protocols:
  MeTR:
    SpeakerDiarization:
      Synthetic:
        train:
          uri: lists/train.txt
          annotation: rttm/{uri}.rttm
          annotated: uem/{uri}.uem
        development:
          uri: lists/val.txt
          annotation: rttm/{uri}.rttm
          annotated: uem/{uri}.uem
""")

# self-check: turns stay in bounds, several speakers, and overlaps do happen
check_rng, overlaps = random.Random(1), 0
for _ in range(20):
    mix, turns = make_conversation(check_rng, pools["train"])
    assert all(0 <= s and s + d <= CONV_S for s, d, _ in turns) and len({spk for *_, spk in turns}) >= 2
    overlaps += sum(a[0] + a[1] > b[0] for a, b in zip(turns, turns[1:]))
assert overlaps > 0, "expected some overlapping speech"
print(f"{len(list((CONV / 'wav').glob('*.wav')))} conversations | example: {len(turns)} turns, "
      f"{sum(d for _, d, _ in turns):.0f}s speech in {CONV_S}s")

840 conversations | example: 13 turns, 51s speech in 60s


## 3. Load the pretrained model

`segmentation-3.0` listens to 10 s windows and marks, frame by frame, which of up to 3 speakers is talking (at most 2 at once). The speakers are anonymous: it learns *when the voice changes*, not *who I am*.

In [4]:
import torch
from lightning.pytorch import Trainer
from lightning.pytorch.callbacks import ModelCheckpoint
from lightning.pytorch.loggers import CSVLogger
from pyannote.audio import Model
from pyannote.audio.tasks import SpeakerDiarization
from pyannote.database import FileFinder, registry

registry.load_database(CONV / "database.yml")
protocol = registry.get_protocol("MeTR.SpeakerDiarization.Synthetic", preprocessors={"audio": FileFinder()})

model = Model.from_pretrained("pyannote/segmentation-3.0")  # uses the token from `hf auth login`
model.task = SpeakerDiarization(protocol, duration=10.0, max_speakers_per_chunk=3, max_speakers_per_frame=2,
                                batch_size=32, num_workers=4)
# pretrained weights only need a nudge: 10x smaller steps than pyannote's default of 1e-3
model.configure_optimizers = lambda: torch.optim.Adam(model.parameters(), lr=1e-4)

'MeTR.SpeakerDiarization.Synthetic' found in /home/darkwarro/Documents/Web_Apps/Exposure Academy/Academy_Exercises/Speaker_Diarization/data/conversations/database.yml does not define the 'scope' of speaker labels (file, database, or global). Setting it to 'file'.


## 4. Fine-tune

The score is **diarization error rate (DER)**: the share of time the model gets wrong (missed speech + false speech + wrong speaker). Lower is better. It is measured on the validation conversations before and after training.

In [5]:
MONITOR = "DiarizationErrorRate"
checkpoint = ModelCheckpoint(dirpath="models", filename="segmentation_tr", monitor=MONITOR, mode="min",
                             save_top_k=1, enable_version_counter=False)
trainer = Trainer(accelerator="auto", max_epochs=10, callbacks=[checkpoint],
                  logger=CSVLogger("logs", name="fine_tuning"), log_every_n_steps=10)

before = trainer.validate(model)[0][MONITOR]
log(f"DER before fine-tuning: {before:.1%}")

trainer.fit(model)

after = checkpoint.best_model_score.item()
log(f"DER before {before:.1%} -> after {after:.1%} | best checkpoint: {checkpoint.best_model_path}")

INFO: GPU available: True (cuda), used: True


18:18:13 GPU available: True (cuda), used: True


INFO: TPU available: False, using: 0 TPU cores


18:18:13 TPU available: False, using: 0 TPU cores


INFO: 💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.


...

Trainable params: 1.5 M                                                                                            
Non-trainable params: 0                                                                                            
Total params: 1.5 M                                                                                                
Total estimated model params size (MB): 5.893                                                                      
Modules in train mode: 30                                                                                          
Modules in eval mode: 0                                                                                            
Total FLOPs: 1.0 B

/home/darkwarro/Documents/Web_Apps/Exposure 
Academy/Academy_Exercises/Speaker_Diarization/.venv/lib/python3.12/site-packages/lightning/pytorch/utilities/data.p
y:123: Your `IterableDataset` has `__len__` defined. In combination with multi-process data loading (when 
num_workers > 1), `__len__` could be inaccurate if each worker is not configured independently to avoid having 
duplicate data.

INFO: `Trainer.fit` stopped: `max_epochs=10` reached.


18:21:21 `Trainer.fit` stopped: `max_epochs=10` reached.


18:21:22 DER before 11.4% -> after 7.8% | best checkpoint: /home/darkwarro/Documents/Web_Apps/Exposure Academy/Academy_Exercises/Speaker_Diarization/models/segmentation_tr.ckpt
